# Ch 10. nanoGPT 스타일 100줄

Ch 8 attention + Ch 9 모던 블록을 한 파일로 모은 GPT-mini 100줄.
Karpathy nanoGPT 정신: *의존성 최소 · 한 화면 안에 모델 전체*.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/desty/study-tiny-llm/blob/main/notebooks/part3/ch10_nanogpt.ipynb)

In [ ]:
# 필요한 경우 설치
# !pip install torch

import torch

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'device: {device}')
print(f'torch version: {torch.__version__}')

## 최소 예제 — GPT-mini 100줄 전체

| 구성 | 줄 수 | 역할 |
|---|---|---|
| `RMSNorm` | 8 | Ch 9 그대로 |
| `apply_rope` | 6 | Ch 9 그대로 |
| `CausalSelfAttention` | 22 | Ch 8 + RoPE |
| `FFN` (SwiGLU) | 10 | Ch 9 그대로 |
| `Block` | 14 | residual 두 번 |
| `GPTMini` | 25 | 전체 |

블록 구조: `x → RMSNorm → Self-Attn → + x → RMSNorm → FFN → + x`

**pre-norm + residual 두 번** — 핵심.

In [ ]:
"""GPT-mini — Karpathy nanoGPT 정신을 따른 단일 파일 구현.
필요: torch (only). 본 책 10M~30M 모델용.
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass


@dataclass
class GPTConfig:
    vocab_size: int = 8000
    n_layer:    int = 6
    n_head:     int = 8
    d_model:    int = 256
    max_len:    int = 512
    dropout:    float = 0.0


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).sqrt()
        return self.gamma * x / (rms + self.eps)


def precompute_rope(head_dim, max_len, base=10000.0, device='cpu'):
    inv = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(max_len, device=device).float()
    freqs = t[:, None] * inv[None, :]
    return freqs.cos(), freqs.sin()                                       # (max_len, head_dim/2)

def apply_rope(x, cos, sin):                                              # (1)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    r1 = x1 * cos - x2 * sin
    r2 = x1 * sin + x2 * cos
    return torch.stack([r1, r2], dim=-1).flatten(-2)


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        assert cfg.d_model % cfg.n_head == 0
        self.n_head, self.head_dim = cfg.n_head, cfg.d_model // cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.dropout = cfg.dropout

    def forward(self, x, cos, sin):
        B, T, D = x.shape
        q, k, v = self.qkv(x).split(D, dim=-1)
        # (B, T, D) → (B, H, T, head_dim)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        # RoPE                                                            (2)
        q = apply_rope(q, cos[:T], sin[:T])
        k = apply_rope(k, cos[:T], sin[:T])
        # SDPA (FlashAttention 자동)
        out = F.scaled_dot_product_attention(
            q, k, v, is_causal=True,
            dropout_p=self.dropout if self.training else 0.0)
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.proj(out)


class FFN(nn.Module):
    """SwiGLU. hidden = (8/3) * dim."""
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        hidden = int(8 * cfg.d_model / 3)
        hidden = ((hidden + 7) // 8) * 8                                  # 8의 배수
        self.w1 = nn.Linear(cfg.d_model, hidden, bias=False)
        self.w2 = nn.Linear(hidden, cfg.d_model, bias=False)
        self.w3 = nn.Linear(cfg.d_model, hidden, bias=False)
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class Block(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn  = CausalSelfAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn   = FFN(cfg)
    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)                        # (3)
        x = x + self.ffn(self.norm2(x))
        return x


class GPTMini(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks  = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm    = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        # weight tying — 입력 임베딩 = 출력 lm_head                       (4)
        self.lm_head.weight = self.tok_emb.weight
        # RoPE 테이블 (학습 안 함)
        cos, sin = precompute_rope(cfg.d_model // cfg.n_head, cfg.max_len)
        self.register_buffer('cos', cos, persistent=False)
        self.register_buffer('sin', sin, persistent=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.max_len
        x = self.tok_emb(idx)                                             # (B, T, D)
        for block in self.blocks:
            x = block(x, self.cos, self.sin)
        x = self.norm(x)
        logits = self.lm_head(x)                                          # (B, T, vocab)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                               targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg.max_len:]                         # (5)
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            if top_k:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, 1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx

print("GPTMini 클래스 정의 완료")

## 실전 — 한 번 돌려보기

확인 포인트:
- 파라미터 수가 **약 10M** — 본 책 기준선
- 학습 전 loss 가 **uniform 분포의 cross-entropy ≈ ln(vocab) = 8.99** — sanity check

In [ ]:
import torch
import math

cfg = GPTConfig(vocab_size=8000, n_layer=6, n_head=8, d_model=320, max_len=512)
model = GPTMini(cfg)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")

# 무작위 입력으로 forward + 손실
x = torch.randint(0, 8000, (2, 64))
y = torch.randint(0, 8000, (2, 64))
logits, loss = model(x, y)
print(f"logits: {logits.shape}, loss: {loss.item():.3f}")  # 학습 전 loss ≈ ln(8000) ≈ 8.99
print(f"expected loss (ln vocab): {math.log(8000):.3f}")

# 생성
prompt = torch.randint(0, 8000, (1, 4))
out = model.generate(prompt, max_new_tokens=20, temperature=0.8, top_k=50)
print("gen shape:", out.shape)  # (1, 24)

In [ ]:
# config 별 파라미터 수 탐색
configs = [
    (6, 256),
    (6, 320),
    (8, 384),
    (12, 512),
    (12, 768),
]

print(f"{'n_layer':>8} {'d_model':>8} {'params (M)':>12}")
print("-" * 32)
for n_layer, d_model in configs:
    cfg_tmp = GPTConfig(vocab_size=8000, n_layer=n_layer, n_head=8, d_model=d_model, max_len=512)
    m = GPTMini(cfg_tmp)
    n_params = sum(p.numel() for p in m.parameters()) / 1e6
    print(f"{n_layer:>8} {d_model:>8} {n_params:>12.2f}")

## 연습문제

1. 위 코드를 그대로 돌려 본인 환경에서 파라미터 수와 학습 전 loss 를 확인하라. ln(8000) ≈ 8.99 와 일치하는가?
2. `n_layer` 를 6 → 12, `d_model` 을 256 → 384 로 늘리면 파라미터가 몇 M 이 되는가? 노트북에서 학습 가능한지 확인.
3. SwiGLU 를 GeLU FFN 으로 교체하고 (`hidden = 4 × d_model`) 파라미터 수와 학습 전 loss 를 비교.
4. `weight_tying` 을 끄고 (`self.lm_head.weight = self.tok_emb.weight` 줄 제거) 파라미터 수가 얼마나 늘어나나?
5. **(생각해볼 것)** nanoGPT 가 의도적으로 의존성을 PyTorch 만으로 한정한 이유는? 학습 자료 vs 프로덕션 코드의 트레이드오프 관점에서 한 단락.

In [ ]:
# 연습문제 1: 파라미터 수와 학습 전 loss 확인 (위 셀로 완료)
# 추가 확인 — vocab_size별 기대 loss
import math
for vocab in [1000, 8000, 32000, 50000]:
    print(f"vocab={vocab:6d}  expected_loss={math.log(vocab):.3f}")

In [ ]:
# 연습문제 2: n_layer=12, d_model=384 로 변경


In [ ]:
# 연습문제 3: SwiGLU → GeLU FFN 으로 교체
class GeLUFFN(nn.Module):
    """표준 GeLU FFN. hidden = 4 * dim."""
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        hidden = 4 * cfg.d_model
        self.fc1 = nn.Linear(cfg.d_model, hidden, bias=False)
        self.fc2 = nn.Linear(hidden, cfg.d_model, bias=False)
    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))

# GeLUFFN 을 Block 에 끼워서 파라미터 수 비교해보기


In [ ]:
# 연습문제 4: weight_tying 끄고 파라미터 수 비교


In [ ]:
# 연습문제 5: nanoGPT 의존성 최소화 철학 — 메모 정리
